# Reproducing Figure 1 of *Understanding Deep Learning Requires Rethinking Generalization* (Zhang et al., ICLR 2017)

This notebook re-implements, from the paper's text alone, the three CIFAR-10 experiments behind Figure 1:

| Panel | Experiment | Paper's setup |
|---|---|---|
| **(a)** learning curves | small Inception trained on true labels, random labels, shuffled pixels, random pixels, Gaussian noise | average training loss vs. thousand steps |
| **(b)** convergence slowdown | Inception, AlexNet, MLP 1x512 trained on labels corrupted with probability *p* ∈ [0, 1] | time to fit, relative to *p* = 0 |
| **(c)** generalization error growth | same runs as (b) | test error after the training set is fit (= generalization error, since training error ≈ 0) |

**Time-to-fit metric (panel b).** A run counts as *fit* at the first step where error on the **full (possibly corrupted) training set**, evaluated in `eval()` mode, is **≤ 1%**. To keep this cheap, a fixed 10k-example training subset is checked every `EVAL_EVERY` steps; whenever the subset error is ≤ 1% the full 50k set is evaluated to confirm. Panel (b) plots `fit_step(p) / fit_step(0)` per model.

**Panel (c)** is the test error measured after training continues for `POST_FIT_EPOCHS` epochs past the fit point (so training error is essentially 0, matching the paper's "after convergence").

### Settings taken from the paper (Appendix A, Sec. 2, Fig. 3)
* CIFAR-10, 50k train / 10k test. Pixels /255, **center crop 28×28**, then **per-image whitening** (TensorFlow `per_image_whitening`: subtract the image mean, divide by `max(std, 1/sqrt(#pixels))`).
* **Small Inception** exactly as in Fig. 3 (Conv module = conv → BatchNorm → ReLU; Inception module = 1×1 ‖ 3×3 branches concatenated; Downsample = stride-2 3×3 conv ‖ 3×3 max-pool concatenated; global 7×7 mean pool; 10-way FC). Parameter count asserted against the paper: **1,649,402**.
* **Small AlexNet**: two (conv 5×5 → max-pool 3×3 → LRN) blocks, FC 384 → FC 192 → 10-way linear. Asserted: **1,387,786**.
* **MLP 1x512**: one hidden layer of 512 ReLU units. Asserted: **1,209,866**.
* SGD, momentum 0.9; initial LR **0.1** (Inception) / **0.01** (AlexNet, MLP); LR multiplied by **0.95 every epoch**.
* **No** weight decay, dropout or data augmentation for all randomization experiments; hyperparameters are *not* changed between true and random labels.
* Corruptions: *partially corrupted labels* (each label independently replaced by a uniform random class with probability *p*; may coincide with the true class), *random labels* (*p* = 1), *shuffled pixels* (one fixed permutation for all train and test images), *random pixels* (an independent permutation per image), *Gaussian* (each image resampled from a Gaussian with the dataset's per-channel mean and variance). All randomizations are drawn once and held fixed across epochs.

### Choices the paper does not specify (made here, flagged so you can change them)
* Batch size **128** (the TensorFlow CIFAR-10 tutorial default of that era). Steps-per-epoch therefore = 391.
* PyTorch default weight initialization; PyTorch default BatchNorm momentum.
* The parameter count of 1,649,402 is matched exactly only if convolutions have **no bias** and BatchNorm has a learned **shift but no scale** (TF's `batch_norm` default); the "w/o BatchNorm" variant in Table 1 has the same count, consistent with the shift becoming the conv bias. That convention is used here.
* LRN uses TF-tutorial constants (window 9, α = 0.001/9 per element, β = 0.75, k = 1). Pooling uses "SAME"-style padding so feature maps go 28 → 14 → 7, which is what reproduces the AlexNet parameter count.
* Mixed precision (fp16 autocast + loss scaling) for speed; set `USE_AMP = False` for strict fp32.
* "Average loss" in panel (a) = mean minibatch training loss over consecutive windows of `LOG_EVERY` steps.

### Running on Kaggle
Accelerator: **GPU T4** (one GPU is used). Turn **Internet on** so torchvision can download CIFAR-10, *or* attach any Kaggle dataset containing the standard `cifar-10-batches-py` folder — the loader searches `/kaggle/input` first.
Every finished run is cached as JSON in `RESULTS_DIR`, so if the session stops you can simply re-run all cells and completed runs are skipped.

Set `MODE` in the next cell: `"smoke"` (≈ minutes, tiny subset, checks the pipeline only — the plot will **not** look like the paper), `"quick"` (6 corruption levels), or `"full"` (11 levels, as in the paper; budget several GPU-hours).

In [ ]:
# ============================ CONFIG ============================
MODE = "full"           # "smoke" | "quick" | "full"
USE_AMP = True          # fp16 autocast on CUDA
BATCH_SIZE = 128        # not stated in the paper
MOMENTUM = 0.9
LR_DECAY_PER_EPOCH = 0.95
INIT_LR = {"inception": 0.1, "alexnet": 0.01, "mlp_1x512": 0.01}
FIT_THRESHOLD = 0.01    # "fit" = full-training-set error <= 1%
SEED = 0

import os
ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else os.path.abspath("./fig1_work")
DATA_DIR = os.path.join(WORK, "data")
RESULTS_DIR = os.path.join(WORK, f"results_{MODE}")
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)

if MODE == "full":
    CORRUPTION_LEVELS = [round(0.1 * i, 1) for i in range(11)]
    PANEL_A_STEPS = 25_000
    MAX_EPOCHS = {"inception": 120, "alexnet": 250, "mlp_1x512": 250}
    POST_FIT_EPOCHS = 5
    N_TRAIN, N_TEST = None, None          # None = full dataset
elif MODE == "quick":
    CORRUPTION_LEVELS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    PANEL_A_STEPS = 25_000
    MAX_EPOCHS = {"inception": 120, "alexnet": 250, "mlp_1x512": 250}
    POST_FIT_EPOCHS = 3
    N_TRAIN, N_TEST = None, None
elif MODE == "smoke":
    CORRUPTION_LEVELS = [0.0, 0.5, 1.0]
    PANEL_A_STEPS = 400
    MAX_EPOCHS = {"inception": 40, "alexnet": 60, "mlp_1x512": 60}
    POST_FIT_EPOCHS = 1
    N_TRAIN, N_TEST = 2000, 1000
else:
    raise ValueError(MODE)

LOG_EVERY = 250 if MODE != "smoke" else 10   # panel (a) loss-averaging window (steps)
EVAL_EVERY = 100 if MODE != "smoke" else 5   # fit check interval (steps)
FIT_SUBSET = 10_000                          # subset used to decide when to run a full-train-set check
print(f"MODE={MODE}  results -> {RESULTS_DIR}")

In [ ]:
import glob, json, math, pickle, tarfile, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    try:
        torch.zeros(1, device=DEVICE) + 1
        print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)
    except Exception as e:
        raise RuntimeError("This PyTorch build cannot run on this GPU (older Kaggle P100 images can hit this). "
                           "Switch the accelerator to 'GPU T4'.") from e
    torch.backends.cudnn.benchmark = True
else:
    print("WARNING: no GPU found - only MODE='smoke' is practical on CPU.")
AMP = USE_AMP and DEVICE.type == "cuda"

## Data: CIFAR-10 → 28×28 center crop → per-image whitening

In [ ]:
def _find_cifar_dir():
    for root in ["/kaggle/input", DATA_DIR]:
        if os.path.isdir(root):
            hits = glob.glob(os.path.join(root, "**", "data_batch_1"), recursive=True)
            if hits:
                return os.path.dirname(hits[0])
    return None

def load_cifar10_raw():
    d = _find_cifar_dir()
    if d is None:
        import torchvision  # needs Kaggle "Internet on"
        torchvision.datasets.CIFAR10(DATA_DIR, train=True, download=True)
        d = _find_cifar_dir()
    def read(names):
        xs, ys = [], []
        for n in names:
            with open(os.path.join(d, n), "rb") as f:
                b = pickle.load(f, encoding="bytes")
            xs.append(b[b"data"]); ys.extend(b[b"labels"])
        x = np.concatenate(xs).reshape(-1, 3, 32, 32)          # uint8, NCHW
        return torch.from_numpy(x), torch.tensor(ys, dtype=torch.long)
    xtr, ytr = read([f"data_batch_{i}" for i in range(1, 6)])
    xte, yte = read(["test_batch"])
    print("CIFAR-10 loaded from", d, "| train", tuple(xtr.shape), "test", tuple(xte.shape))
    return xtr, ytr, xte, yte

def to_unit_cropped(x_uint8):
    # /255 into [0,1], then central 28x28 crop (offset 2 on each side)
    return x_uint8.float().div_(255.0)[:, :, 2:30, 2:30].contiguous()

def per_image_whiten(x):
    # TensorFlow per_image_whitening: (x - mean) / max(std, 1/sqrt(num_elements)), per image
    n = x.shape[0]
    flat = x.reshape(n, -1)
    mean = flat.mean(1, keepdim=True)
    std = flat.std(1, unbiased=False, keepdim=True)
    adj = torch.clamp(std, min=1.0 / math.sqrt(flat.shape[1]))
    return ((flat - mean) / adj).reshape_as(x)

XTR_RAW, YTR_TRUE, XTE_RAW, YTE_TRUE = load_cifar10_raw()
if N_TRAIN: XTR_RAW, YTR_TRUE = XTR_RAW[:N_TRAIN], YTR_TRUE[:N_TRAIN]
if N_TEST:  XTE_RAW, YTE_TRUE = XTE_RAW[:N_TEST],  YTE_TRUE[:N_TEST]
XTR_UNIT, XTE_UNIT = to_unit_cropped(XTR_RAW), to_unit_cropped(XTE_RAW)   # [0,1], (N,3,28,28)

## Randomizations (Section 2.1)
Every variant is generated once with a fixed seed and then held fixed for the whole run (labels and pixels are *consistent across epochs*, which is what makes memorization possible).

In [ ]:
def corrupt_labels(y, p, seed):
    # Partially corrupted labels: independently with prob p, replace by a uniform random class.
    g = torch.Generator().manual_seed(seed)
    mask = torch.rand(len(y), generator=g) < p
    rnd = torch.randint(0, 10, (len(y),), generator=g)
    return torch.where(mask, rnd, y)

def shuffle_pixels_fixed(xs, seed):
    # One random permutation of pixel locations (RGB kept together), applied to all train & test images.
    g = torch.Generator().manual_seed(seed)
    n_pix = xs[0].shape[2] * xs[0].shape[3]
    perm = torch.randperm(n_pix, generator=g)
    return [x.reshape(x.shape[0], 3, n_pix)[:, :, perm].reshape_as(x).contiguous() for x in xs]

def random_pixels_per_image(x, seed, chunk=10_000):
    # A different random permutation of pixel locations for every image.
    g = torch.Generator().manual_seed(seed)
    n, c, h, w = x.shape
    flat = x.reshape(n, c, h * w)
    out = torch.empty_like(flat)
    for i in range(0, n, chunk):
        idx = torch.rand(min(chunk, n - i), h * w, generator=g).argsort(1)
        out[i:i + chunk] = flat[i:i + chunk].gather(2, idx[:, None, :].expand(-1, c, -1))
    return out.reshape_as(x)

def gaussian_images(n, ref_train, seed):
    # Pixels i.i.d. Gaussian with the per-channel mean/std of the original (cropped, [0,1]) training images.
    g = torch.Generator().manual_seed(seed)
    mu = ref_train.mean(dim=(0, 2, 3)).view(1, 3, 1, 1)
    sd = ref_train.std(dim=(0, 2, 3)).view(1, 3, 1, 1)
    return torch.randn(n, *ref_train.shape[1:], generator=g) * sd + mu

def make_dataset(kind, p=0.0, seed=SEED):
    # Returns whitened train/test tensors on DEVICE. Test labels are always the true labels.
    xtr, xte, ytr = XTR_UNIT, XTE_UNIT, YTR_TRUE
    if kind == "true_labels":
        pass
    elif kind == "corrupt_labels":
        ytr = corrupt_labels(YTR_TRUE, p, seed + 1)
    elif kind == "random_labels":
        ytr = corrupt_labels(YTR_TRUE, 1.0, seed + 1)
    elif kind == "shuffled_pixels":
        xtr, xte = shuffle_pixels_fixed([XTR_UNIT, XTE_UNIT], seed + 2)
    elif kind == "random_pixels":
        xtr, xte = random_pixels_per_image(XTR_UNIT, seed + 3), random_pixels_per_image(XTE_UNIT, seed + 4)
    elif kind == "gaussian":
        xtr, xte = gaussian_images(len(XTR_UNIT), XTR_UNIT, seed + 5), gaussian_images(len(XTE_UNIT), XTR_UNIT, seed + 6)
    else:
        raise ValueError(kind)
    return (per_image_whiten(xtr).to(DEVICE), ytr.to(DEVICE),
            per_image_whiten(xte).to(DEVICE), YTE_TRUE.to(DEVICE))

# quick checks of the randomizations
_y = corrupt_labels(YTR_TRUE, 0.5, 123)
print(f"p=0.5 -> fraction of labels changed: {(_y != YTR_TRUE).float().mean():.3f} (expected 0.5*0.9 = 0.45)")
_s = shuffle_pixels_fixed([XTR_UNIT[:4]], 7)[0]
assert torch.allclose(_s.sum(dim=(2, 3)), XTR_UNIT[:4].sum(dim=(2, 3)), atol=1e-4)  # permutation preserves content

## Architectures (Fig. 3 and Appendix A), with parameter counts checked against Table 1

In [ ]:
class ConvModule(nn.Module):
    # Fig. 3 "Conv Module": convolution -> batch norm -> ReLU.
    # BN with learned shift only (TF batch_norm default, no scale); the conv then needs no bias.
    # Without BN the conv carries a bias instead -> identical parameter count (as in Table 1).
    def __init__(self, cin, cout, k, stride=1, bn=True):
        super().__init__()
        self.conv = nn.Conv2d(cin, cout, k, stride=stride, padding=k // 2, bias=not bn)
        self.bn = nn.BatchNorm2d(cout, affine=False) if bn else None
        self.beta = nn.Parameter(torch.zeros(cout)) if bn else None
    def forward(self, x):
        x = self.conv(x)
        if self.bn is not None:
            x = self.bn(x) + self.beta.view(1, -1, 1, 1)
        return F.relu(x)

class InceptionModule(nn.Module):
    def __init__(self, cin, ch1, ch3, bn=True):
        super().__init__()
        self.b1 = ConvModule(cin, ch1, 1, bn=bn)
        self.b3 = ConvModule(cin, ch3, 3, bn=bn)
    def forward(self, x):
        return torch.cat([self.b1(x), self.b3(x)], 1)

class DownsampleModule(nn.Module):
    def __init__(self, cin, ch3, bn=True):
        super().__init__()
        self.conv = ConvModule(cin, ch3, 3, stride=2, bn=bn)
        self.pool = nn.MaxPool2d(3, stride=2, padding=1)
    def forward(self, x):
        return torch.cat([self.conv(x), self.pool(x)], 1)

class SmallInception(nn.Module):
    def __init__(self, bn=True, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            ConvModule(3, 96, 3, bn=bn),                                    # 28x28
            InceptionModule(96, 32, 32, bn), InceptionModule(64, 32, 48, bn),
            DownsampleModule(80, 80, bn),                                   # -> 14x14, 160 ch
            InceptionModule(160, 112, 48, bn), InceptionModule(160, 96, 64, bn),
            InceptionModule(160, 80, 80, bn), InceptionModule(160, 48, 96, bn),
            DownsampleModule(144, 96, bn),                                  # -> 7x7, 240 ch
            InceptionModule(240, 176, 160, bn), InceptionModule(336, 176, 160, bn),
        )
        self.fc = nn.Linear(336, num_classes)
    def forward(self, x):
        x = self.features(x)
        return self.fc(x.mean(dim=(2, 3)))                                  # global 7x7 mean pool

class SmallAlexNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        lrn = lambda: nn.LocalResponseNorm(size=9, alpha=0.001, beta=0.75, k=1.0)  # = TF depth_radius=4, alpha=0.001/9
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 5, padding=2), nn.ReLU(inplace=True), nn.MaxPool2d(3, 2, padding=1), lrn(),   # 14x14
            nn.Conv2d(64, 64, 5, padding=2), nn.ReLU(inplace=True), nn.MaxPool2d(3, 2, padding=1), lrn(),  # 7x7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64 * 7 * 7, 384), nn.ReLU(inplace=True),
            nn.Linear(384, 192), nn.ReLU(inplace=True), nn.Linear(192, num_classes))
    def forward(self, x):
        return self.classifier(self.features(x))

class MLP(nn.Module):
    def __init__(self, hidden=(512,), in_dim=3 * 28 * 28, num_classes=10):
        super().__init__()
        layers, d = [nn.Flatten()], in_dim
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU(inplace=True)]; d = h
        layers.append(nn.Linear(d, num_classes))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

def make_model(name):
    return {"inception": lambda: SmallInception(bn=True),
            "inception_nobn": lambda: SmallInception(bn=False),
            "alexnet": SmallAlexNet,
            "mlp_1x512": lambda: MLP((512,)),
            "mlp_3x512": lambda: MLP((512, 512, 512))}[name]()

PAPER_PARAMS = {"inception": 1_649_402, "inception_nobn": 1_649_402, "alexnet": 1_387_786,
                "mlp_1x512": 1_209_866, "mlp_3x512": 1_735_178}
for _n, _p in PAPER_PARAMS.items():
    _m = make_model(_n)
    _c = sum(t.numel() for t in _m.parameters())
    _o = _m(torch.zeros(2, 3, 28, 28))
    assert _o.shape == (2, 10)
    assert _c == _p, f"{_n}: {_c} != paper {_p}"
    print(f"{_n:15s} params = {_c:,}  (paper: {_p:,})  OK")

## Training engine
Plain SGD+momentum, LR × 0.95 at the start of every epoch (staircase), no explicit regularization. The whole dataset lives on the GPU and minibatches are sampled from a fresh permutation each epoch.

In [ ]:
def _prep(model_name, x):
    return x.contiguous(memory_format=torch.channels_last) if not model_name.startswith("mlp") else x

@torch.no_grad()
def error_rate(model, model_name, x, y, bs=2500):
    was_training = model.training
    model.eval()
    wrong = torch.zeros((), device=x.device)
    for i in range(0, len(x), bs):
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
            out = model(_prep(model_name, x[i:i + bs]))
        wrong += (out.argmax(1) != y[i:i + bs]).sum()
    model.train(was_training)
    return wrong.item() / len(x)

def train(model_name, data, *, max_steps=None, max_epochs=None, stop_at_fit=False,
          post_fit_epochs=0, log_every=LOG_EVERY, eval_every=EVAL_EVERY, seed=SEED, tag=""):
    xtr, ytr, xte, yte = data
    torch.manual_seed(seed)
    model = make_model(model_name).to(DEVICE)
    if not model_name.startswith("mlp"):
        model = model.to(memory_format=torch.channels_last)
    lr0 = INIT_LR[model_name.replace("_nobn", "")]
    opt = torch.optim.SGD(model.parameters(), lr=lr0, momentum=MOMENTUM, weight_decay=0.0)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)

    n = len(xtr)
    spe = math.ceil(n / BATCH_SIZE)                              # steps per epoch
    total = max_steps if max_steps is not None else max_epochs * spe
    g = torch.Generator(device="cpu").manual_seed(seed + 99)
    sub_idx = torch.randperm(n, generator=g)[:min(FIT_SUBSET, n)].to(DEVICE)

    hist = {"loss_step": [], "loss": [], "train_err_step": [], "train_err_subset": []}
    loss_acc = torch.zeros((), device=DEVICE); loss_cnt = 0
    fit_step, stop_step, diverged = None, total, False
    step, t0 = 0, time.time()
    model.train()
    while step < stop_step:
        epoch = step // spe
        for grp in opt.param_groups:
            grp["lr"] = lr0 * (LR_DECAY_PER_EPOCH ** epoch)
        perm = torch.randperm(n, device=DEVICE)
        for i in range(spe):
            idx = perm[i * BATCH_SIZE:(i + 1) * BATCH_SIZE]
            xb, yb = _prep(model_name, xtr[idx]), ytr[idx]
            with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
                loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            loss_acc += loss.detach().float(); loss_cnt += 1
            step += 1

            if step % log_every == 0:
                avg = (loss_acc / loss_cnt).item()
                hist["loss_step"].append(step); hist["loss"].append(avg)
                loss_acc.zero_(); loss_cnt = 0
                if not math.isfinite(avg):
                    diverged = True; stop_step = step; break

            if stop_at_fit and fit_step is None and step % eval_every == 0:
                sub_err = error_rate(model, model_name, xtr[sub_idx], ytr[sub_idx])
                hist["train_err_step"].append(step); hist["train_err_subset"].append(sub_err)
                if sub_err <= FIT_THRESHOLD and error_rate(model, model_name, xtr, ytr) <= FIT_THRESHOLD:
                    fit_step = step
                    stop_step = min(total, step + post_fit_epochs * spe)
            if step >= stop_step:
                break

    res = {"model": model_name, "tag": tag, "steps": step, "epochs": step / spe, "steps_per_epoch": spe,
           "fit_step": fit_step, "fit_epoch": None if fit_step is None else fit_step / spe,
           "diverged": diverged,
           "final_train_err": error_rate(model, model_name, xtr, ytr),
           "final_test_err": error_rate(model, model_name, xte, yte),
           "wall_s": time.time() - t0, "hist": hist}
    fs = "not fit" if fit_step is None else f"fit at step {fit_step} ({res['fit_epoch']:.1f} ep)"
    print(f"  [{model_name} {tag}] {fs} | train err {res['final_train_err']:.4f} | "
          f"test err {res['final_test_err']:.4f} | {res['wall_s']/60:.1f} min")
    return res

def cached(name, fn):
    path = os.path.join(RESULTS_DIR, name + ".json")
    if os.path.exists(path):
        with open(path) as f:
            r = json.load(f)
        print(f"  [cached] {name}")
        return r
    r = fn()
    with open(path, "w") as f:
        json.dump(r, f)
    return r

## Panel (a): small Inception on CIFAR-10 under five settings
Fixed budget of `PANEL_A_STEPS` steps, same hyperparameters for every setting.

In [ ]:
PANEL_A_KINDS = ["true_labels", "random_labels", "shuffled_pixels", "random_pixels", "gaussian"]
panel_a = {}
for kind in PANEL_A_KINDS:
    print(f"Panel (a): {kind}")
    def _run(kind=kind):
        data = make_dataset(kind)
        r = train("inception", data, max_steps=PANEL_A_STEPS, tag=kind)
        del data; torch.cuda.empty_cache()
        return r
    panel_a[kind] = cached(f"a_inception_{kind}", _run)

## Panels (b) and (c): label-corruption sweep for Inception, AlexNet and MLP 1x512
Each run trains until the full training set error is ≤ 1% (time to fit), then continues `POST_FIT_EPOCHS` epochs and records the test error.
The same corruption seed is used across models, so all three networks see the same corrupted labels at each *p*.

In [ ]:
SWEEP_MODELS = ["inception", "alexnet", "mlp_1x512"]
panel_bc = {m: {} for m in SWEEP_MODELS}
for p in CORRUPTION_LEVELS:
    data = None
    for m in SWEEP_MODELS:
        name = f"bc_{m}_p{p:.1f}"
        print(f"Sweep: {m}, p={p:.1f}")
        def _run(m=m, p=p):
            global data
            if data is None:
                data = make_dataset("corrupt_labels", p=p)
            return train(m, data, max_epochs=MAX_EPOCHS[m], stop_at_fit=True,
                         post_fit_epochs=POST_FIT_EPOCHS, tag=f"p={p:.1f}")
        panel_bc[m][p] = cached(name, _run)
    del data; torch.cuda.empty_cache()

## Figure 1
This cell only needs the cached JSON results, so it can be re-run on its own.

In [ ]:
import os, json, glob
import numpy as np
import matplotlib.pyplot as plt

def load_results(results_dir):
    a, bc = {}, {}
    for path in glob.glob(os.path.join(results_dir, "*.json")):
        name = os.path.basename(path)[:-5]
        with open(path) as f:
            r = json.load(f)
        if name.startswith("a_inception_"):
            a[name[len("a_inception_"):]] = r
        elif name.startswith("bc_"):
            m, p = name[3:].rsplit("_p", 1)
            bc.setdefault(m, {})[float(p)] = r
    return a, bc

def plot_figure1(results_dir, out_prefix):
    a, bc = load_results(results_dir)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

    # (a) learning curves
    sty_a = {"true_labels": ("true labels", "tab:blue", "s"), "random_labels": ("random labels", "tab:red", "o"),
             "shuffled_pixels": ("shuffled pixels", "tab:green", "*"), "random_pixels": ("random pixels", "m", None),
             "gaussian": ("gaussian", "k", "D")}
    ax = axes[0]
    for k, (lab, c, mk) in sty_a.items():
        if k not in a: continue
        h = a[k]["hist"]
        s = np.asarray(h["loss_step"]) / 1000.0
        ax.plot(s, h["loss"], color=c, marker=mk, markersize=4, markevery=max(1, len(s) // 25), label=lab)
    ax.set_xlabel("thousand steps"); ax.set_ylabel("average_loss"); ax.set_ylim(bottom=0)
    ax.set_title("(a) learning curves (Inception)"); ax.legend(); ax.grid(alpha=0.3)

    sty_bc = {"inception": ("Inception", "k", "s"), "alexnet": ("AlexNet", "tab:green", "o"),
              "mlp_1x512": ("MLP 1x512", "tab:blue", "*")}
    # (b) relative time to fit; unfinished runs shown as open markers at their step budget (a lower bound)
    ax = axes[1]
    for m, (lab, c, mk) in sty_bc.items():
        runs = bc.get(m, {})
        if 0.0 not in runs or runs[0.0]["fit_step"] is None: continue
        base = runs[0.0]["fit_step"]
        ps = sorted(runs)
        ys = [(runs[p]["fit_step"] if runs[p]["fit_step"] is not None else runs[p]["steps"]) / base for p in ps]
        ax.plot(ps, ys, color=c, marker=mk, label=lab)
        miss = [(p, y) for p, y in zip(ps, ys) if runs[p]["fit_step"] is None]
        if miss:
            ax.scatter(*zip(*miss), s=90, facecolors="none", edgecolors="r", zorder=5)
    ax.set_xlabel("label corruption"); ax.set_ylabel("time to fit (relative to p = 0)")
    ax.set_title("(b) convergence slowdown"); ax.legend(); ax.grid(alpha=0.3)

    # (c) test error after fitting
    ax = axes[2]
    for m, (lab, c, mk) in sty_bc.items():
        runs = bc.get(m, {})
        if not runs: continue
        ps = sorted(runs)
        ax.plot(ps, [runs[p]["final_test_err"] for p in ps], color=c, marker=mk, label=lab)
    ax.axhline(0.9, color="r", ls="--", lw=1.5)
    ax.set_xlabel("label corruption"); ax.set_ylabel("test_error"); ax.set_ylim(0.1, 1.0)
    ax.set_title("(c) generalization error growth"); ax.legend(); ax.grid(alpha=0.3)

    fig.suptitle("Fitting random labels and random pixels on CIFAR-10 (reproduction of Zhang et al. 2017, Fig. 1)")
    fig.tight_layout()
    fig.savefig(out_prefix + ".png", dpi=150); fig.savefig(out_prefix + ".pdf")
    plt.show()
    return a, bc

panel_a_res, panel_bc_res = plot_figure1(RESULTS_DIR, os.path.join(WORK, f"figure1_{MODE}"))

## Checks against the paper's qualitative claims
These print PASS/WARN rather than asserting, so the figure is always produced. Reference numbers are from the paper's Table 1 (no augmentation, no weight decay) and Section 2.

In [ ]:
def check(ok, msg): print(("PASS  " if ok else "WARN  ") + msg)

a, bc = panel_a_res, panel_bc_res
if a:
    for k, r in a.items():
        check(r["hist"]["loss"][-1] < 0.05, f"(a) {k}: final average loss {r['hist']['loss'][-1]:.3f} (paper: ~0 for every setting)")
    def first_below(r, thr=0.5):
        for s, l in zip(r["hist"]["loss_step"], r["hist"]["loss"]):
            if l < thr: return s
        return float("inf")
    if all(k in a for k in ["true_labels", "random_labels", "random_pixels", "gaussian"]):
        t = {k: first_below(a[k]) for k in a}
        print("    steps until avg loss < 0.5:", t)
        check(t["true_labels"] < t["random_labels"], "(a) true labels fit faster than random labels")
        check(max(t["random_pixels"], t["gaussian"]) < t["random_labels"],
              "(a) random pixels / gaussian start converging before random labels (paper Sec. 2.1)")

paper_test_acc = {"inception": 85.75, "alexnet": 76.07, "mlp_1x512": 50.51}
paper_rand_acc = {"inception": 9.78, "alexnet": 9.86, "mlp_1x512": 10.61}
for m, runs in bc.items():
    ps = sorted(runs)
    unfit = [p for p in ps if runs[p]["fit_step"] is None]
    check(not unfit, f"(b) {m}: every corruption level reached <=1% train error" + (f" (not fit at p={unfit})" if unfit else ""))
    if 0.0 in runs:
        check(abs(100 * (1 - runs[0.0]["final_test_err"]) - paper_test_acc[m]) < 6,
              f"(c) {m} p=0 test acc {100*(1-runs[0.0]['final_test_err']):.1f}% (paper {paper_test_acc[m]}%)")
    if 1.0 in runs:
        check(abs(100 * (1 - runs[1.0]["final_test_err"]) - paper_rand_acc[m]) < 2,
              f"(c) {m} p=1 test acc {100*(1-runs[1.0]['final_test_err']):.1f}% (paper {paper_rand_acc[m]}%, chance 10%)")
    if 0.0 in runs and 1.0 in runs and runs[0.0]["fit_step"] and runs[1.0]["fit_step"]:
        check(runs[1.0]["fit_step"] / runs[0.0]["fit_step"] > 1.3,
              f"(b) {m} random labels take {runs[1.0]['fit_step']/runs[0.0]['fit_step']:.2f}x longer than true labels "
              "(paper: small constant factor, ~1.5-3.6x)")
    errs = [runs[p]["final_test_err"] for p in ps]
    check(all(e2 >= e1 - 0.02 for e1, e2 in zip(errs, errs[1:])), f"(c) {m}: test error rises with corruption")

print("\nPer-run summary (fit epoch, final train err, test err):")
for m, runs in bc.items():
    for p in sorted(runs):
        r = runs[p]
        fe = "  -  " if r["fit_epoch"] is None else f"{r['fit_epoch']:5.1f}"
        print(f"  {m:10s} p={p:.1f}  fit_epoch={fe}  train_err={r['final_train_err']:.4f}  test_err={r['final_test_err']:.4f}")